# neural-cost: free GPU benchmark run

Runs the neural-cost benchmark suite on a free cloud accelerator and packages
the results so they can be committed to `benchmarks/results/<device>/`.

| Where | How to get the accelerator | What runs |
|---|---|---|
| **Google Colab** | *Runtime → Change runtime type → T4 GPU* | PyTorch, JAX and TensorFlow on CUDA |
| **Kaggle** | *Settings → Accelerator → GPU T4 x2* or *GPU P100* (needs a phone-verified account) | PyTorch, JAX and TensorFlow on CUDA |

Every result file records the neural-cost version, the **git commit**, the
framework versions and the device, so a run can always be traced to the code
that produced it.

**Before you start:** make sure nothing else is running in the session; the
free GPUs are not shared with other users, but the notebook's own other cells
are. Run the cells top to bottom (*Runtime → Run all*).

In [ ]:
# ---- Settings ---------------------------------------------------------------
# A release tag (e.g. "v0.7.0"), a branch, or a full commit hash.
REF = "main"
REPO = "https://github.com/davidgraymi/neural-cost.git"

QUICK = False          # True: fewer batch sizes and repeats (~5 min instead of ~30)
RUN_VALIDATION = True  # lower-bound check, zero-shot prediction, gap attribution
RUN_GPU_SWEEP = True   # architectures x frameworks x batch sizes on the accelerator
RUN_CPU_SWEEP = False  # the host CPU (slow on free tiers; needed only for --crossover)

In [ ]:
import os, pathlib, shutil, subprocess, sys

FAILURES = []  # (command, exit code) of every command that failed

def sh(cmd, check=True):
    """Run cmd, streaming its output into the cell.

    Child processes write to the kernel's own stdout, which Colab and Kaggle do
    not show in the cell, so the output is piped through print() instead.
    """
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    returncode = proc.wait()
    if returncode:
        FAILURES.append((cmd, returncode))
        print(f"!! exit code {returncode}: {cmd}", flush=True)
        if check:
            raise RuntimeError(f"command failed ({returncode}): {cmd}")
    return returncode

# Colab images now ship a /kaggle directory too, so check Kaggle's own variable.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
SRC = WORK / "neural-cost"

HAS_CUDA = shutil.which("nvidia-smi") is not None and sh("nvidia-smi -L", check=False) == 0
print("CUDA:", HAS_CUDA, " Kaggle:", ON_KAGGLE)
if not HAS_CUDA:
    print("WARNING: no CUDA GPU found; change the runtime type first or results will be CPU-only.")

In [ ]:
# ---- Get the code at REF and install it ------------------------------------
if SRC.exists():
    shutil.rmtree(SRC)
sh(f"git clone --quiet {REPO} {SRC}")
sh(f"git -C {SRC} checkout --quiet {REF}")
COMMIT = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
print("Benchmarking commit", COMMIT)

# The preinstalled CUDA builds of torch and TensorFlow are kept; JAX needs its
# CUDA wheel.  Each benchmark runs in its own process, so the three
# frameworks never share one interpreter.
sh(f"pip install --quiet -e '{SRC}[torch,tensorflow]' matplotlib")
if HAS_CUDA:
    sh("pip install --quiet -U 'jax[cuda12]'")
else:
    sh("pip install --quiet -U jax")

In [ ]:
# ---- Record the environment -------------------------------------------------
os.chdir(SRC)
os.environ["NEURAL_COST_COMMIT"] = COMMIT  # survives even if .git is stripped
# `pip install -e` ran inside this kernel, so its .pth file (which puts src/ on
# the path) is only read by new processes.  Without src/ here, the skill file
# the install copies into site-packages/neural_cost/ imports as an empty
# namespace package.
sys.path[:0] = [str(SRC / "src"), str(SRC)]
from benchmarks.run_info import run_metadata
import json
print(json.dumps(run_metadata("(probe)"), indent=2))
sh("python -c 'from neural_cost import detect_hardware; s, d = detect_hardware(); print(s, d.source)'")

In [ ]:
# ---- Run --------------------------------------------------------------------
quick = " --quick" if QUICK else ""
import time
STARTED = time.time()  # result files written (or rewritten) after this are new
FAILURES.clear()

if RUN_VALIDATION:
    backends = "torch-cuda torch-cpu jax-cpu" if HAS_CUDA else "torch-cpu jax-cpu"
    sh(f"python benchmarks/validate.py{quick} --backends {backends}", check=False)

if RUN_GPU_SWEEP and HAS_CUDA:
    sh(f"python benchmarks/collect_gpu_data.py{quick} --device cuda", check=False)

if RUN_CPU_SWEEP:
    sh(f"python benchmarks/collect_data.py{quick}", check=False)
    if RUN_GPU_SWEEP and HAS_CUDA:
        sh(f"python benchmarks/collect_gpu_data.py{quick} --device cuda --crossover", check=False)

if FAILURES:
    print("\nFailed commands (their output is above):")
    for cmd, code in FAILURES:
        print(f"  exit {code}: {cmd}")
else:
    print("\nAll benchmark commands succeeded.")

In [ ]:
# ---- Reports ----------------------------------------------------------------
results = SRC / "benchmarks" / "results"
written = sorted({p.parent for p in results.glob("*/*") if p.stat().st_mtime >= STARTED})
if not written:
    failed = "".join(f"\n  exit {code}: {cmd}" for cmd, code in FAILURES) or " none"
    raise RuntimeError(
        "No result files were written, so there is nothing to package. "
        f"Failed commands:{failed}\nSee the run cell's output for the errors."
    )
for folder in written:
    if (folder / "benchmark_gpu_data.json").exists():
        sh(f"python benchmarks/generate_gpu_report.py --results-dir {folder}", check=False)
    if (folder / "benchmark_data.json").exists():
        sh(f"python benchmarks/generate_report.py --results-dir {folder}", check=False)
    print(folder.name, "->", sorted(p.name for p in folder.iterdir()))

In [ ]:
# ---- Package ----------------------------------------------------------------
import zipfile

archive = WORK / f"neural-cost-results-{COMMIT[:12]}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in written:
        for path in folder.rglob("*"):
            zf.write(path, path.relative_to(results))
print("Wrote", archive)
if not ON_KAGGLE:
    from google.colab import files
    files.download(str(archive))
else:
    print("Download it from the notebook's Output panel.")

## Committing the results

Unzip the archive into `benchmarks/results/` of a checkout **at the same
commit** (or any later one) and open a pull request:

```bash
unzip neural-cost-results-<commit>.zip -d benchmarks/results/
git add benchmarks/results/<device>/
git commit -m "chore(benchmarks): <device> results for <commit>"
```

Check the run line at the top of each report: it names the commit that was
measured. Results from a `git_dirty` or unknown commit should not be published.